In [58]:
import os
import glob
import re
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import lr_scheduler, Adam
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from PIL import Image

Variables

In [59]:
IMAGE_PATH = "/content/drive/MyDrive/cecum_t1_a/color"
IMAGE_DEPTH_PATH = "/content/drive/MyDrive/cecum_t1_a/depth"
RANDOM_SEED = 123
BATCH_SIZE = 4
MAX_EPOCHS = 20

TRAIN_RATIO = 0.7
VAL_RATIO = (1.0 - TRAIN_RATIO)/2
TEST_RATIO = VAL_RATIO

Helper functions

In [74]:
def resize_predictions(predictions, ground_truths):
  """
  Resizes the model predictions to ground truths size. Result is used for calculating
  loss function.
  """
  resized = torch.nn.functional.interpolate(predictions.unsqueeze(1),
                                            mode='bicubic',
                                            size = ground_truths.shape[-2:],
                                            align_corners=False).squeeze(1)
  return resized

def scale_invariant_loss(prediction, target):
  """
  A scale-invariant loss function specifically for MiDaS.
  """
  mask = target > 0 # filter out all invalid pixels
  prediction = prediction[mask]
  target = target[mask]
  M = mask.sum().float()

  # normalize the pixel values for target and prediction
  # adding a small value to standard deviation to prevent division by 0
  prediction_norm = (prediction - prediction.mean()) / (prediction.std() + 1e-6)
  target_norm = (target - target.mean()) / (target.std() + 1e-6)

  # Loss function
  loss = 1.0 / (2. * M) * torch.sum(torch.abs(prediction_norm - target_norm))
  return loss

def get_image_pairs(img_folder: str, depth_folder: str):
  """
  The function gets a pair of the image and its depth map image and returns a
  list of dictionaries.
  """
  # Pre-emptive checks
  if os.path.exists(img_folder) and os.path.exists(depth_folder):
    img_files   = glob.glob(os.path.join(img_folder, '*.png'))
    depth_files = glob.glob(os.path.join(depth_folder, '*.tiff'))
    img_folder_count   = len(img_files)
    depth_folder_count = len(depth_files)
    # print("""
    # images: {},
    # depth images: {}
    # """.format(img_folder_count,depth_folder_count))

  else:
    print('One or both files do not exist.')

  if img_folder_count != depth_folder_count:
    print('Mismatch in number of images found in files. {} missing'.format(abs(img_folder_count - depth_folder_count)))

  # get the image pairs
  depth_dict = {}
  for path in depth_files:
    filename = os.path.basename(path)
    _match = re.match(r'^(\d{4})_', filename)
    if _match:
      depth_dict[_match.group(1)] = path

  pairs = list()
  unmatched = list()

  for color_path in sorted(img_files):
    filename = os.path.basename(color_path)
    image_match = re.match(r'^(\d+)_', filename)
    if image_match:
      image_prefix = image_match.group(1).zfill(4)
      if image_prefix in depth_dict:
        pairs.append({
            'color': color_path,
            'depth': depth_dict[image_prefix]
        })
      else:
        unmatched.append(filename)

  if unmatched:
    print('No depth match found for {} color image (s): {}'.format(len(unmatched), unmatched))

  print('Successfully paired {} images'.format(len(pairs)))
  return pairs

def train_val_test_split(pairs: list, random_state = RANDOM_SEED, train_ratio=TRAIN_RATIO,
                         val_ratio = VAL_RATIO, test_ratio=TEST_RATIO):
  """
  Function creates training, validation and test split for image pairs.

  Returns a dictionary for such split.
  """
  train_pairs, temp_pairs = train_test_split(pairs, test_size= test_ratio + val_ratio, random_state = RANDOM_SEED)

  val_pairs, test_pairs = train_test_split(temp_pairs, test_size = 0.5, random_state = RANDOM_SEED)

  splits = {
      'train': train_pairs,
      'val': val_pairs,
      'test': test_pairs
  }

  return splits

In [61]:
class DepthDataset(Dataset):
  def __init__(self, pairs: list, transform=None):
    self.transform = transform
    self.pairs = pairs

  def __len__(self):
    return len(self.pairs)

  def __getitem__(self, idx):
    pair = self.pairs[idx]
    color_image = np.array(Image.open(pair['color']).convert('RGB'))
    depth_image = np.array(Image.open(pair['depth'])) # keep in grayscale

    # Use MiDaS transformation for the images
    if self.transform:
      color_image = self.transform(color_image).squeeze(0)

    return {
        'color': color_image,
        'depth_gt': torch.tensor(depth_image, dtype=torch.float32)
    }

Create splits

In [62]:
image_pairs  = get_image_pairs(IMAGE_PATH, IMAGE_DEPTH_PATH)

# get splits
splits = train_val_test_split(image_pairs)

Train = splits['train']
Val = splits['val']
Test = splits['test']

Successfully paired 276 images


MiDaS Model

In [63]:
MODEL_TYPE = 'DPT_Large' # can either be "DPT_Hybrid" or "MiDaS_Small"
MIDAS_HYBRID_MODEL = torch.hub.load("intel-isl/MiDaS", MODEL_TYPE)

Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Moving model to GPU if available

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
MIDAS_HYBRID_MODEL.to(device)

Applying appropriate image preprocessing transforms

In [65]:
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")

if MODEL_TYPE == "DPT_Large" or MODEL_TYPE == "DPT_Hybrid":
    transform = midas_transforms.dpt_transform
else:
    transform = midas_transforms.small_transform

Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Freezing MiDas pre-trained layers

In [66]:
for param in MIDAS_HYBRID_MODEL.parameters():
  param.requires_grad = False
for param in MIDAS_HYBRID_MODEL.scratch.parameters():
  param.requires_grad = True

Preparring the data

In [67]:
Train_dataset = DepthDataset(Train, transform=transform)
Val_dataset = DepthDataset(Val, transform=transform)
Test_dataset = DepthDataset(Test, transform=transform)

# Create data loader
Train_loader = DataLoader(Train_dataset, batch_size = BATCH_SIZE, shuffle=False)
Val_loader = DataLoader(Val_dataset, batch_size = BATCH_SIZE, shuffle=False)
Test_loader = DataLoader(Test_dataset, batch_size = BATCH_SIZE, shuffle=False)

Set the optimizer and learning rate scheduler

In [68]:
# optimizer
optimizer = Adam(filter(lambda p: p.requires_grad, MIDAS_HYBRID_MODEL.parameters()),
                 lr = 1e-4)

# Learning rate scheduler
learning_rate_scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)
# learning_rate_scheduler = lr_scheduler.StepLR(optimizer, step_size = 0.01)

Training the data

In [69]:
batch_iter = iter(Train_loader)
first_batch = next(batch_iter)
first_batch['color'].shape

torch.Size([4, 3, 384, 480])

In [79]:
best_val_loss = float('inf')

for epoch in range(MAX_EPOCHS):
  # train the model
  training_loss = 0.
  MIDAS_HYBRID_MODEL.train()

  for batch in Train_loader:
    color_tensors = batch['color'].to(device)
    depth_groundTruths = batch['depth_gt'].to(device)
    optimizer.zero_grad()
    predictions = MIDAS_HYBRID_MODEL(color_tensors)

    # resize the predictions to match ground truth data
    predictions = resize_predictions(predictions, depth_groundTruths)

    # get loss value
    loss = scale_invariant_loss(predictions, depth_groundTruths)
    loss.backward()
    optimizer.step()

    training_loss += loss.item()

  average_training_loss = training_loss / len(Train_loader)
  print("The average training loss: {}".format(round(average_training_loss*100, 4))

  # validation
  val_loss = 0.
  with torch.no_grad():
    for batch in Val_loader:
      color_tensors = batch['color'].to(device)
      depth_groundTruths = batch['depth_gt'].to(device)

      predictions = MIDAS_HYBRID_MODEL(color_tensors)
      predictions = resize_predictions(predictions, depth_groundTruths)

      loss = scale_invariant_loss(predictions, depth_groundTruths)
      val_loss += loss.item()

  average_validation_loss =  val_loss/ len(Val_loader)
  learning_rate_scheduler.step(average_validation_loss)

  print(f"Epoch [{epoch +1 }/{MAX_EPOCHS}] "
        f"Training loss: {round(average_training_loss * 100, 4)} "
        f"Validation loss: {round(average_validation_loss *100 , 4)} "
        f"LR: {learning_rate_scheduler.get_last_lr()}")

  if average_validation_loss < best_val_loss:
    best_val_loss = average_validation_loss
    torch.save(MIDAS_HYBRID_MODEL.state_dict(), "/content/drive/MyDrive/midas_finetuned_best.pth")
    print(f"The best model saved has a validation loss: {round(best_val_loss*100,4)}")

The average trainingloss: 0.021148884935038432
Epoch [1/20] Training loss: 2.1149 Validation loss: 2.3155 LR: [0.0001]
The best model saved has a validation loss: 2.3155
The average trainingloss: 0.018543442007990515
Epoch [2/20] Training loss: 1.8543 Validation loss: 1.8879 LR: [0.0001]
The best model saved has a validation loss: 1.8879
The average trainingloss: 0.017328250659059505
Epoch [3/20] Training loss: 1.7328 Validation loss: 1.6805 LR: [0.0001]
The best model saved has a validation loss: 1.6805
The average trainingloss: 0.016091690170673693
Epoch [4/20] Training loss: 1.6092 Validation loss: 1.6119 LR: [0.0001]
The best model saved has a validation loss: 1.6119
The average trainingloss: 0.015093787478245035
Epoch [5/20] Training loss: 1.5094 Validation loss: 1.5488 LR: [0.0001]
The best model saved has a validation loss: 1.5488
The average trainingloss: 0.01461329172384374
Epoch [6/20] Training loss: 1.4613 Validation loss: 1.556 LR: [0.0001]
The average trainingloss: 0.01386

We can consider performing cross validation on k=5 for hyperparameter tuning on:
* Optimizer
* Freezing Strategy (Fine tune on scratch layer vs scratch + last 2 encoder blocks)
* Learning Rate